# Squareprompt Capital ML baseline v2

A causal, per-market ridge regression predicts the return from the next open through the fourth following close. Features use completed OHLCV bars only. Targets enter training only after their entire four-bar outcome is observed. Feature scaling and regularization selection use training observations only.

The first fit uses at least 32 matured examples; subsequent fits occur after 14 elapsed days of market time, using a trailing 90-day window capped at 2,048 examples. Insufficient history means flat exposure. Coefficients stay fixed between fits; indicators update each bar. Each market and evaluation has fresh, independent model state.

Signals account for the pinned round-trip fee/spread and adverse funding, use a no-trade band and forecast uncertainty, and apply volatility sizing, exposure limits, turnover hysteresis and a latched drawdown breaker. This is a research starting point, not evidence of an edge.

Every stage receives completed history and returns state. `training_history` is the only permitted fitting dataset. During search it is the available historical prefix; during final holdout it ends before the embargo. Holdout observations may update causal indicators but never fit coefficients, preprocessing, calibration or hyperparameters. `bar_interval_ms` and pinned `execution_costs` are supplied by the trusted evaluator. Model artifacts and fit events are attached to evaluation outputs. No filesystem, network or future access inside strategy cells.


In [ ]:
def stage_sources(history, state):
    state["source_count"] = len(state.get("sources", {}))
    return state


In [ ]:
import numpy as np

def stage_quality(history, state):
    prices = np.asarray(history[["open", "high", "low", "close"]], dtype=float)
    if not np.isfinite(prices).all() or (prices <= 0).any():
        raise ValueError("OHLC prices must be finite and positive")
    if (prices[:, 1] < np.maximum(prices[:, 0], prices[:, 3])).any() or (prices[:, 2] > np.minimum(prices[:, 0], prices[:, 3])).any():
        raise ValueError("Inconsistent OHLC candle")
    if not history["time"].is_monotonic_increasing or history["time"].duplicated().any():
        raise ValueError("Candle times must be unique and increasing")
    volume = np.asarray(history["volume"], dtype=float)
    if not np.isfinite(volume).all() or (volume < 0).any():
        raise ValueError("Volume must be finite and nonnegative")
    interval = state.get("bar_interval_ms", 3600000)
    state["tradable"] = len(history) < 2 or int(history["time"].iloc[-1] - history["time"].iloc[-2]) <= 1.5 * interval
    return state


In [ ]:
def stage_storage(history, state):
    state["as_of"] = int(history["available_at"].iloc[-1])
    state["observed_bars"] = len(history)
    return state


In [ ]:
import math

def stage_state(history, state):
    returns = history["close"].pct_change()
    state["volatility"] = max(0.000001, float(returns.tail(24).std()))
    slow_vol = max(0.000001, float(returns.tail(96).std()))
    state["volatility_ratio"] = state["volatility"] / slow_vol
    interval = state.get("bar_interval_ms", 3600000)
    state["bars_per_year"] = 365.0 * 24 * 3600000 / interval
    return state


In [ ]:
import numpy as np
import pandas as pd

model_cache = {}
FEATURE_NAMES = ["return_1", "momentum_3", "momentum_12", "momentum_24", "volatility_6", "volatility_24", "candle_range", "candle_body", "volume_surprise", "trend_distance"]

def ml_features(history):
    close = history["close"]
    returns = close.pct_change()
    volume = history["volume"]
    features = pd.DataFrame({
        "return_1": returns,
        "momentum_3": close.pct_change(3),
        "momentum_12": close.pct_change(12),
        "momentum_24": close.pct_change(24),
        "volatility_6": returns.rolling(6).std(),
        "volatility_24": returns.rolling(24).std(),
        "candle_range": (history["high"] - history["low"]) / close,
        "candle_body": close / history["open"] - 1,
        "volume_surprise": volume / volume.rolling(24).mean().clip(lower=0.000001) - 1,
        "trend_distance": close / close.rolling(24).mean() - 1,
    })
    return np.asarray(features[FEATURE_NAMES], dtype=float)

def ridge_fit(x, y, alpha):
    mean = np.mean(x, axis=0)
    scale = np.maximum(np.std(x, axis=0), 0.000001)
    z = np.clip((x - mean) / scale, -8, 8)
    intercept = float(np.mean(y))
    weights = np.linalg.solve(z.T @ z + alpha * np.eye(z.shape[1]), z.T @ (y - intercept))
    return mean, scale, weights, intercept

def stage_models(history, state):
    now = int(state["as_of"])
    horizon = int(state.get("prediction_horizon", 4))
    training = state.get("training_history", history)
    window = int(state.get("training_window", 2048))
    since = now - int(state.get("training_days", 90)) * 86400000
    training = training[training["available_at"] >= since].tail(window + 24 + horizon)
    retrain_interval_ms = int(state.get("retrain_interval_ms") or 0)
    refit_due = not model_cache or (
        retrain_interval_ms > 0 and now - model_cache["fitted_at"] >= retrain_interval_ms
    )
    state["refit_due"] = refit_due
    if refit_due and len(training) > 24 + horizon:
        x = ml_features(training)[:-horizon]
        close = np.asarray(training["close"], dtype=float)
        opening = np.asarray(training["open"], dtype=float)
        # Feature j predicts close[j+horizon] / open[j+1] - 1.
        # Both target endpoints are already completed; no negative shift is used.
        y = close[horizon:] / opening[1:len(opening) - horizon + 1] - 1
        valid = np.isfinite(x).all(axis=1) & np.isfinite(y)
        x, y = x[valid][-window:], y[valid][-window:]
        trained_through = int(training["available_at"].iloc[-1])
        if len(y) >= int(state.get("minimum_training_rows", 32)) and trained_through > model_cache.get("trained_through", 0):
            alpha = 10.0
            validation_rmse = None
            # A chronological tail of past training data chooses regularization.
            # Its scaler is fitted solely on the preceding training fold.
            if len(y) >= 80:
                split = int(len(y) * 0.8)
                train_end = split - horizon  # Purge overlapping forward targets.
                best_error = None
                for candidate in [1.0, 10.0, 100.0]:
                    mean, scale, weights, intercept = ridge_fit(x[:train_end], y[:train_end], candidate)
                    pred = np.clip((x[split:] - mean) / scale, -8, 8) @ weights + intercept
                    error = float(np.mean((pred - y[split:]) ** 2))
                    if best_error is None or error < best_error:
                        alpha, best_error = candidate, error
                validation_rmse = float(np.sqrt(best_error))
            mean, scale, weights, intercept = ridge_fit(x, y, alpha)
            fitted = np.clip((x - mean) / scale, -8, 8) @ weights + intercept
            residual = max(0.000001, float(np.sqrt(np.mean((fitted - y) ** 2))))
            event = {"fitted_at": now, "trained_through": trained_through, "rows": len(y), "alpha": alpha,
                "mean": mean.tolist(), "scale": scale.tolist(), "weights": weights.tolist(),
                "intercept": intercept, "uncertainty": max(residual, validation_rmse or residual)}
            events = model_cache.get("events", []) + [event]
            model_cache.update({"mean": mean, "scale": scale, "weights": weights, "intercept": intercept,
                "uncertainty": max(residual, validation_rmse or residual), "fitted_at": now,
                "trained_through": trained_through, "rows": len(y), "alpha": alpha, "events": events,
                "prediction_horizon": horizon, "last_feature_time": int(training["time"].iloc[-horizon - 1])})
    state["model_ready"] = bool(model_cache)
    state["forecast"] = 0.0
    state["forecast_uncertainty"] = 1.0
    if model_cache:
        feature = ml_features(history.tail(128))[-1]
        if np.isfinite(feature).all():
            z = (feature - model_cache["mean"]) / model_cache["scale"]
            prediction = float(np.clip(z, -8, 8) @ model_cache["weights"] + model_cache["intercept"])
            uncertainty = model_cache["uncertainty"]
            state["forecast"] = max(-3 * uncertainty, min(3 * uncertainty, prediction))
            state["forecast_uncertainty"] = uncertainty
            state["feature_outlier_fraction"] = float(np.mean(np.abs(z) > 4))
        else:
            state["model_ready"] = False
    if model_cache:
        state["model_artifact"] = {"kind": "ridge", "version": "baseline-ml-v2", "features": FEATURE_NAMES,
            "mean": model_cache["mean"].tolist(), "scale": model_cache["scale"].tolist(),
            "weights": model_cache["weights"].tolist(), "intercept": model_cache["intercept"],
            "uncertainty": model_cache["uncertainty"], "rows": model_cache["rows"], "alpha": model_cache["alpha"],
            "fitted_at": model_cache["fitted_at"], "trained_through": model_cache["trained_through"],
            "prediction_horizon": model_cache["prediction_horizon"], "fit_events": model_cache["events"],
            "retrain_interval_ms": retrain_interval_ms,
            "bar_interval_ms": state.get("bar_interval_ms", 3600000), "feature_clip": 8, "prediction_clip_sigma": 3}
    return state


In [ ]:
def stage_signals(history, state):
    forecast = state.get("forecast", 0.0)
    costs = state.get("execution_costs", {})
    round_trip = 2 * (costs.get("taker_rate", 0.0) + costs.get("half_spread_rate", 0.0))
    holding_hours = state.get("prediction_horizon", 4) * state.get("bar_interval_ms", 3600000) / 3600000
    funding = costs.get("funding_rate_hourly", 0.0)
    adverse_funding = max(0.0, funding if forecast >= 0 else -funding) * holding_hours
    threshold = round_trip + adverse_funding + 0.10 * state.get("forecast_uncertainty", 1.0)
    state["entry_threshold"] = threshold
    state["target"] = 1.0 if forecast > threshold else (-1.0 if forecast < -threshold else 0.0)
    if not state.get("model_ready", True) or not state.get("tradable", True):
        state["target"] = 0.0
    state["confidence"] = min(1.0, abs(forecast) / max(0.000001, state.get("forecast_uncertainty", 1.0)))
    return state


In [ ]:
import math

def stage_sizing(history, state):
    annual_volatility_target = 0.15
    bar_target = annual_volatility_target / math.sqrt(state.get("bars_per_year", 8760))
    weight = min(0.8, bar_target / max(0.000001, state.get("volatility", 0.01)))
    state["target"] *= weight * state.get("confidence", 1.0)
    previous = state.get("previous_position", 0.0)
    # Keep small same-direction allocation changes from paying repeated fees.
    if state["target"] * previous > 0 and abs(state["target"] - previous) < 0.03:
        state["target"] = previous
    return state


In [ ]:
def stage_exposure(history, state):
    cap = min(0.8, state.get("max_exposure", 1.0))
    state["target"] = max(-cap, min(cap, state["target"]))
    return state


In [ ]:
risk_memory = {"halted": False}

def stage_risk(history, state):
    if state["drawdown"] > 0.15:
        risk_memory["halted"] = True
    if risk_memory["halted"] or not state.get("tradable", True):
        state["target"] = 0.0
    state["risk_halted"] = risk_memory["halted"]
    return state


In [ ]:
def stage_retraining(history, state):
    state["training_window"] = 2048
    state["training_days"] = 90
    state["minimum_training_rows"] = 32
    state["prediction_horizon"] = 4
    state["retrain_interval_ms"] = 14 * 86400000
    return state


In [ ]:
def stage_drift(history, state):
    if state.get("volatility_ratio", 1.0) > 2.0 or state.get("feature_outlier_fraction", 0.0) > 0.2:
        state["target"] *= 0.5
    return state


In [ ]:
def stage_registry(history, state):
    state["pipeline_version"] = "baseline-ml-v2"
    return state
